# Session 1 · Getting Started with Generative AI

Generative AI › Track 2 · Gen AI Accelerator

Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Runtime ▸ Run all in Colab). Outputs below were produced by actually executing this notebook.

### Example · `session-01_next_word.py`

In [1]:
"""A tiny 'language model': count which word follows which (a bigram model),
then predict the next word. Real LLMs do the same job - predict the next token -
but with billions of learned parameters instead of a word-count table."""
from collections import Counter, defaultdict

corpus = """students learn python in the lab . students learn ai in the class .
teachers explain ai in the class . students ask questions in the lab .
ai helps students learn faster . teachers use ai to plan the class ."""
words = corpus.split()

follows = defaultdict(Counter)
for a, b in zip(words, words[1:]):
    follows[a][b] += 1

def next_word_probs(word):
    c = follows[word]
    total = sum(c.values())
    return {w: n / total for w, n in c.most_common()}

for w in ["students", "in", "ai"]:
    probs = next_word_probs(w)
    print(f"after '{w}':", ", ".join(f"{k} {v:.2f}" for k, v in probs.items()))

# Greedy generation: always pick the most likely next word
word, out = "teachers", ["teachers"]
for _ in range(6):
    word = follows[word].most_common(1)[0][0]
    out.append(word)
print("\ngreedy text:", " ".join(out))

after 'students': learn 0.75, ask 0.25
after 'in': the 1.00
after 'ai': in 0.50, helps 0.25, to 0.25

greedy text: teachers explain ai in the class .


### Example · `session-01_temperature.py`

In [2]:
"""How 'temperature' reshapes a model's next-token probabilities.
Low temperature -> safe, repetitive; high temperature -> varied, riskier."""
import math

logits = {"class": 2.0, "lab": 1.5, "library": 0.5, "moon": -1.0}  # raw model scores

def softmax(scores, t):
    exps = {k: math.exp(v / t) for k, v in scores.items()}
    z = sum(exps.values())
    return {k: e / z for k, e in exps.items()}

print("Next word after 'Students study in the ...'\n")
print(f"{'temp':>5} " + " ".join(f"{k:>8}" for k in logits))
for t in [0.2, 0.7, 1.0, 1.5]:
    p = softmax(logits, t)
    print(f"{t:>5} " + " ".join(f"{p[k]:>8.2f}" for k in logits))

Next word after 'Students study in the ...'

 temp    class      lab  library     moon
  0.2     0.92     0.08     0.00     0.00
  0.7     0.62     0.30     0.07     0.01
  1.0     0.53     0.32     0.12     0.03
  1.5     0.45     0.32     0.17     0.06
